In [ ]:
import os
from ase.io import read, write
import numpy as np

atoms = read("/home/loharkar/QuEnAIS-quantum-embedding/data/raw/1CA2.pdb")

zn_idx = [i for i,a in enumerate(atoms) if a.symbol=="Zn"][0]
zn_pos = atoms[zn_idx].position

sel = [i for i,a in enumerate(atoms)
       if np.linalg.norm(a.position-zn_pos) < 5.0]

cluster = atoms[sel]
write("cluster.xyz", cluster)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

smiles = {
    "L1": "c1ncc[nH]1",
    "L2": "Cc1ncc[nH]1",
    "L3": "c1ccncc1"
}

for name,smi in smiles.items():
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    AllChem.UFFOptimizeMolecule(mol)

    Chem.MolToXYZFile(mol, f"{name}.xyz")

In [ ]:
from ase.io import read, write

cluster = read("cluster.xyz")
lig = read("L1.xyz")

lig.translate([0,0,2.0])  # move near metal

combined = cluster + lig
write("L1_complex.xyz", combined)

In [ ]:
from pyscf import gto, scf

mol = gto.Mole()
mol.atom = open("L1_complex.xyz").read()
mol.basis = 'sto-3g'
mol.build()

mf = scf.RHF(mol)
E = mf.kernel()

print(E)

In [ ]:
import numpy as np
from pyscf import gto, scf

def clean_extended_xyz(input_file, output_file="clean.xyz"):
    with open(input_file, 'r') as f:
        lines = f.readlines()
    
    natoms = int(lines[0].strip())
    # Skip the header line (line 1)
    atom_lines = lines[2:2 + natoms]   # the actual atom lines
    
    with open(output_file, 'w') as f:
        f.write(f"{natoms}\n")
        f.write("Cleaned for PySCF\n")
        for line in atom_lines:
            parts = line.split()
            # Take only: element (parts[0]), x, y, z (parts[1:4])
            clean_line = f"{parts[0]}  {parts[1]}  {parts[2]}  {parts[3]}\n"
            f.write(clean_line)

# Use it once
clean_extended_xyz("L1_complex.xyz")

# Now run your calculation on the clean file
mol = gto.Mole()
mol.atom = "clean.xyz"          # or mol.atom = open("clean.xyz").read()
mol.basis = 'sto-3g'
mol.build()

mf = scf.(mol)
E = mf.kernel()
print("Energy:", E)

In [ ]:
from pyscf.gto import Mole
from pyscf.lib import logger
from asf.wrapper import find_from_mol, sized_space_from_mol
# Find one active space.
active_space = find_from_mol(mol)

print(active_space)

In [ ]:
from pyscf import gto, scf, dft
from asf.wrapper import find_from_scf   # preferred

# ... your mol definition with charge=2, spin=0 ...

# First get a good mean-field yourself (much more reliable than ASF's internal one)
mf = scf.RHF(mol)
mf.max_cycle = 150
mf.level_shift = 0.25
mf.damp = 0.35
mf.init_guess = 'huckel'
mf = mf.density_fit()          # helps a lot
mf.kernel()

# If RHF still struggles, fall back to DFT (very common and accepted for embedding workflows)
if not mf.converged:
    print("RHF failed → using DFT")
    mf = dft.RKS(mol)
    mf.xc = 'b3lyp'            # or 'pbe0'
    mf = mf.density_fit()
    mf.kernel()

# Now feed the converged mf to ASF
nel, mo_list, mo_coeff = find_from_scf(
    mf,
    entropy_threshold=0.15,     # 0.1–0.2 is typical; lower = bigger active space
    mp2_max_orb=25,             # limit the MP2 window to speed it up (important for your size)
    # states=1,                 # or [(0,0)] for singlet if needed
)

print(f"Active space: {nel} electrons in {len(mo_list)} orbitals")
print("Active MO indices:", mo_list)

In [ ]:
mol.charge = 2
mol.spin = 0
mol.build()

# Optional: pre-run a quick RHF and overwrite the guess (sometimes helps)
mf_guess = scf.RHF(mol).run()
mol = mf_guess.mol   # not always necessary

active_space = find_from_mol(
    mol,
    # You can pass some kwargs in newer versions; check your ASF version
)